### LangChain Token Tracking & Memory
This notebook demonstrates how to build a stateful AI assistant using LangChain. We will cover:

API Key Management using a .env file.

Conversation Memory so the AI remembers past interactions.

Token & Cost Tracking to monitor exactly how much each request costs.

### Code for Configuration
LangChain Token Tracking & Session Memory (2026 Edition)
This notebook demonstrates the modern LCEL (LangChain Expression Language) approach.

langchain-openai: Handles the connection to GPT-4o.

langchain-community: Provides the callback utilities for cost tracking.

langchain-core: Manages the message history and "runnable" logic.

In [7]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.callbacks import get_openai_callback

# Load environment variables from .env file
load_dotenv()

# Setup the model
# Using 'gpt-4o' which is highly efficient for 2026 workflows
model = ChatOpenAI(model="gpt-4o", temperature=0.5)

### Memory Architecture
In modern LangChain, we don't use the old ConversationBufferMemory class directly in a chain. Instead, we use a History Factory. This allows the same program to handle multiple users (sessions) simultaneously by storing their unique message histories in a dictionary.

In [8]:
# 1. Define the Prompt with a placeholder for 'chat_history'
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant who remembers user details."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

# 2. Build the chain using the pipe (|) operator
chain = prompt | model

# 3. Define the session storage
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 4. Wrap the chain with history management
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

### Execute with Token Tracking

In [12]:
def chat(user_text, session_id="user_123"):
    # The callback listener specifically for OpenAI usage metadata
    with get_openai_callback() as cb:
        config = {"configurable": {"session_id": session_id}}
        response = chain_with_memory.invoke({"input": user_text}, config=config)
        
        print(f"🤖 AI: {response.content}")
        print("\n" + "="*40)
        print(f"📊 USAGE FOR SESSION: {session_id}")
        print(f"   - Total Tokens: {cb.total_tokens}")
        print(f"   - Prompt Tokens: {cb.prompt_tokens}")
        print(f"   - Completion Tokens: {cb.completion_tokens}")
        print(f"   - Estimated Cost: ${cb.total_cost:.6f}")
        print("="*40 + "\n")

# Call 1: Establish identity
chat("Hi, my name is Alex and I'm a chef.")

# Call 2: Verify memory
chat("What is my profession?")

# Call 3: 
chat("What is the name of the most popular dish in Italian cuisine?")

🤖 AI: Hello Alex! It's always a pleasure to hear from you. How have things been in the kitchen? Are there any new culinary projects you're excited about?

📊 USAGE FOR SESSION: user_123
   - Total Tokens: 408
   - Prompt Tokens: 378
   - Completion Tokens: 30
   - Estimated Cost: $0.001245

🤖 AI: You are a chef.

📊 USAGE FOR SESSION: user_123
   - Total Tokens: 426
   - Prompt Tokens: 421
   - Completion Tokens: 5
   - Estimated Cost: $0.001102

🤖 AI: One of the most popular dishes in Italian cuisine is pizza, especially the classic Margherita, which is topped with tomatoes, mozzarella cheese, and fresh basil. Additionally, pasta dishes like spaghetti alla carbonara and lasagna are also incredibly popular and representative of Italian culinary tradition.

📊 USAGE FOR SESSION: user_123
   - Total Tokens: 502
   - Prompt Tokens: 447
   - Completion Tokens: 55
   - Estimated Cost: $0.001668

